In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('datasets')

finance_df = pd.read_json(DATA_DIR / 'finance.jsonl', lines=True)
medicine_df = pd.read_json(DATA_DIR / 'medicine.jsonl', lines=True)
open_qa_df = pd.read_json(DATA_DIR / 'open_qa.jsonl', lines=True)
reddit_eli5_df = pd.read_json(DATA_DIR / 'reddit_eli5.jsonl', lines=True)
wiki_csai_df = pd.read_json(DATA_DIR / 'wiki_csai.jsonl', lines=True)

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

def df_to_text_label(df, source_name):
    human = (
        df[['human_answers']]
        .explode('human_answers')
        .dropna()
        .rename(columns={'human_answers': 'text'})
    )
    human['label'] = 1

    ai = (
        df[['chatgpt_answers']]
        .explode('chatgpt_answers')
        .dropna()
        .rename(columns={'chatgpt_answers': 'text'})
    )
    ai['label'] = 0

    human['source'] = source_name
    ai['source'] = source_name

    return pd.concat([human, ai], ignore_index=True)

datasets = {
    "finance": df_to_text_label(finance_df, "finance"),
    "medicine": df_to_text_label(medicine_df, "medicine"),
    "open_qa": df_to_text_label(open_qa_df, "open_qa"),
    "reddit_eli5": df_to_text_label(reddit_eli5_df, "reddit_eli5"),
    "wiki_csai": df_to_text_label(wiki_csai_df, "wiki_csai"),
}

for name, df in datasets.items():
    print(name, df.shape)

finance (8436, 3)
medicine (2585, 3)
open_qa (4748, 3)
reddit_eli5 (67996, 3)
wiki_csai (1684, 3)


In [3]:
results = []

for test_name, test_df in datasets.items():
    train_domain = [df for name, df in datasets.items() if name != test_name]
    train_df = pd.concat(train_domain, ignore_index=True)

    X_train, y_train = train_df['text'], train_df['label']
    X_test, y_test = test_df['text'], test_df['label']

    clf = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=200000,
            ngram_range=(1, 2),
            min_df=2
        )),
        ("logreg", LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
        )),
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results.append({"test_domain": test_name, "accuracy": acc})

    print(f"=== Leave out {test_name} (train on the other 4) ===")
    print(f"Accuracy: {acc:.4f}")
    print()

avg_acc = sum(r["accuracy"] for r in results) / len(results)
print("Average accuracy over all leave-one-domain-out runs:", avg_acc)
results

=== Leave out finance (train on the other 4) ===
Accuracy: 0.9339

=== Leave out medicine (train on the other 4) ===
Accuracy: 0.9760

=== Leave out open_qa (train on the other 4) ===
Accuracy: 0.6763

=== Leave out reddit_eli5 (train on the other 4) ===
Accuracy: 0.9067

=== Leave out wiki_csai (train on the other 4) ===
Accuracy: 0.7987

Average accuracy over all leave-one-domain-out runs: 0.8583098227480812


[{'test_domain': 'finance', 'accuracy': 0.933854907539118},
 {'test_domain': 'medicine', 'accuracy': 0.9760154738878143},
 {'test_domain': 'open_qa', 'accuracy': 0.676284751474305},
 {'test_domain': 'reddit_eli5', 'accuracy': 0.9067003941408318},
 {'test_domain': 'wiki_csai', 'accuracy': 0.7986935866983373}]

### Interpreting Results

From the above cell we can see that the logistic regression model trained is able to generalize across domains. However we should note that when the `open_qa` and `wiki_csai` datasets are left out of training, the accuracy score is observably worse. Additionally, the largest bulk of our dataset (`reddit_eli5`) being left out still resulted in a high accuracy score.

As such, we conclude that the most important datasets that contain features that can identify chatgpt generated text are the `open_qa` and `wiki_csai` datasets.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

full_df = pd.concat(datasets.values(), ignore_index=True)

for ngram_range in [(1,1), (1,2), (1,3)]:
    temp_vec = TfidfVectorizer(
        min_df=2,
        ngram_range=ngram_range
    )
    temp_vec.fit(full_df['text'])
    print(f"ngram_range={ngram_range} → {len(temp_vec.vocabulary_):,} unique tokens")

ngram_range=(1, 1) → 50,968 unique tokens
ngram_range=(1, 2) → 770,772 unique tokens
ngram_range=(1, 3) → 2,029,460 unique tokens


In [9]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report

# Combine all domains
full_df = pd.concat(datasets.values(), ignore_index=True)
X_all, y_all = full_df['text'], full_df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("logreg", LogisticRegression(max_iter=1000, n_jobs=-1)),
])

param_grid = {
    "tfidf__max_features": [200000, 300000, 500000],
    "tfidf__ngram_range":  [(1, 1), (1, 2)],
    "logreg__C":           [0.1, 1.0, 10.0],
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)

# Tune only on training portion
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
[CV] END logreg__C=0.1, tfidf__max_features=200000, tfidf__ngram_range=(1, 1); total time=   6.9s
[CV] END logreg__C=0.1, tfidf__max_features=200000, tfidf__ngram_range=(1, 1); total time=   7.5s
[CV] END logreg__C=0.1, tfidf__max_features=200000, tfidf__ngram_range=(1, 1); total time=   7.6s
[CV] END logreg__C=0.1, tfidf__max_features=200000, tfidf__ngram_range=(1, 1); total time=   7.5s
[CV] END logreg__C=0.1, tfidf__max_features=200000, tfidf__ngram_range=(1, 1); total time=   7.7s
[CV] END logreg__C=0.1, tfidf__max_features=300000, tfidf__ngram_range=(1, 1); total time=   7.4s
[CV] END logreg__C=0.1, tfidf__max_features=300000, tfidf__ngram_range=(1, 1); total time=   7.2s
[CV] END logreg__C=0.1, tfidf__max_features=300000, tfidf__ngram_range=(1, 1); total time=   7.2s
[CV] END logreg__C=0.1, tfidf__max_features=300000, tfidf__ngram_range=(1, 1); total time=   7.3s
[CV] END logreg__C=0.1, tfidf__max_features=300000, tfidf

In [10]:
print("Best params:", grid_search.best_params_)
print("Best CV F1: ", grid_search.best_score_)

final_model = grid_search.best_estimator_
y_pred = final_model.predict(X_test)
print("\nFinal Test Set Performance:")
print(classification_report(y_test, y_pred))

Best params: {'logreg__C': 10.0, 'tfidf__max_features': 300000, 'tfidf__ngram_range': (1, 2)}
Best CV F1:  0.9867780587884909

Final Test Set Performance:
              precision    recall  f1-score   support

           0       0.98      0.96      0.97      5381
           1       0.98      0.99      0.99     11709

    accuracy                           0.98     17090
   macro avg       0.98      0.98      0.98     17090
weighted avg       0.98      0.98      0.98     17090

